Let's start off with a basic node class.
We will suggest you hold the data in a np.array
Then, create grad so that it is a numpy array of zeros like data.
This is so that we can leverage numpy's np.zeros_like() function to copy all types of inputs. Let's also keep track of the prior nodes (parents)

In [15]:
#2 mins for this: Transition 1

import numpy as np

class Node():
    def __init__(self, data, _parents=(), label = "_"):
        """
        We want to initialize data, grad, and backward.
        where:
            data is an array, do we have data?
            grad is an array, do we have gradients?
            _prev is a set, which one of our inputs holds the previous nodes?
        """
        #TODO:  # Store data as NumPy array, # Gradient initialized to zero,  # Parent nodes in the computation graph
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self._parents = set(_parents)
        self.backward_fn = lambda: None

        self.label = label

    def __repr__(self):
        return f"Node({self.label} | data={self.data}, grad={self.grad})"

In [16]:

# 5 mins for this: Transition 2

def add_node(a_node, b_node):
    """
    Element-wise addition: A + B
    """

    output = Node(
        a_node.data + b_node.data,
        {a_node, b_node}
    )

    def add_backward():
        a_node.grad += output.grad
        b_node.grad += output.grad

    output.backward_fn = add_backward

    return output

def mul_node(a_node, b_node):
    """
    Element-wise multiplication: A * B
    """
    # TODO:
    # Produce a new node:
    #   Make sure the output's data is correct
    #   Make sure the output knows what its parents are
    output = Node(a_node.data * b_node.data, {a_node, b_node})
   
    # Note:
    # - out.grad will be populated later
    # - for now, assume out.grad has its proper value for this node.
    #
    # TODO:
    # define a backward function to update a_node's gradients and b_node's gradients
    # based on out.grad
    #
    def mul_backward():
        a_node.grad += b_node.data * output.grad
        b_node.grad += a_node.data * output.grad
    
    output.backward_fn = mul_backward
    return output

In [17]:

print("Testing Node Initialization...")
a_test = Node(3.0)
b_test = Node(2.0)

assert a_test.data == 3.0, "Error: a_test.data not set correctly"
assert b_test.data == 2.0, "Error: b_test.data not set correctly"
assert a_test.grad == 0.0, "Error: a_test.grad not set correctly"
assert b_test.grad == 0.0, "Error: b_test.grad not set correctly"

print("Passed!")

print("Testing Node Addition...")
c_test = add_node(a_test, b_test)
assert c_test.data == 5.0, "Error: add_node DATA not evaluated correctly"
assert c_test.grad == 0.0, "Error: add_node GRAD not initialized correctly"
print("Passed!")

print("Testing Node Addition Backward...")
c_test.grad = 1.0
c_test.backward_fn()
assert a_test.grad == 1.0, "Error: add_node BACKWARD does not update parent gradients correctly"
assert b_test.grad == 1.0, "Error: add_node BACKWARD does not update parent gradients correctly"

print("Passed!")

a_test = Node(3.0)
b_test = Node(2.0)

print("Testing Node Multiplication...")
d_test = mul_node(a_test, b_test)
assert d_test.data == 6.0, "Error: mul_node DATA not evaluated correctly"
assert d_test.grad == 0.0, "Error: mul_node GRAD not initialized correctly"

print("Passed!")


print("Testing Node Multiplication Backward...")
d_test.grad = 1.0
d_test.backward_fn()
assert a_test.grad == 2.0, "Error: mul_node BACKWARD does not update parent gradients correctly"
assert b_test.grad == 3.0, "Error: mul_node BACKWARD does not update parent gradients correctly"

print("Passed!")

print("All tests passed!")



Testing Node Initialization...
Passed!
Testing Node Addition...
Passed!
Testing Node Addition Backward...
Passed!
Testing Node Multiplication...
Passed!
Testing Node Multiplication Backward...
Passed!
All tests passed!


In [18]:
#2 mins for this: Transition 2

def matmul_node(a_node, b_node):
    """Matrix multiplication: A @ B"""
    out = Node(a_node.data @ b_node.data, {a_node, b_node})
    
    def matmul_backward():
        a_node.grad += out.grad @ b_node.data.T
        b_node.grad += a_node.data.T @ out.grad
    
    out.backward_fn = matmul_backward
    
    return out

In [19]:
a_test = Node(np.array([[1, 2], [3, 4]]))
b_test = Node(np.array([[2, 0], [1, 2]]))

print("Testing Matrix Multiplication...")
c_test = matmul_node(a_test, b_test)

assert np.allclose(c_test.data, np.array([[4, 4], [10, 8]])), "Error: matmul DATA not evaluated correctly"
print("Passed!")

print("Testing Matrix Multiplication Backward...")
c_test.grad = np.array([[1, 1], [1, 1]])
c_test.backward_fn()

assert np.allclose(a_test.grad, np.array([[2, 3], [2, 3]])), "Error: matmul BACKWARD does not update parent gradients correctly"
assert np.allclose(b_test.grad, np.array([[4, 4], [6, 6]])), "Error: matmul BACKWARD does not update parent gradients correctly"
print("Passed!")

print("All tests passed!")

Testing Matrix Multiplication...
Passed!
Testing Matrix Multiplication Backward...
Passed!
All tests passed!


In [20]:
# 5 mins for this: Transition 2

class Node(Node):
    def backward(self):
        backward_list = []

        self.grad = np.ones_like(self.data)  # Placeholder for the gradient of the loss

        #TODO: Define a recursive function to build the backward list
        def build_backward_list(n):
            if n not in backward_list:
                for parent in n._parents:
                    build_backward_list(parent)
                backward_list.append(n)

        build_backward_list(self)
        # TODO: print the output backward list if you want to see it!
        print(backward_list)
        # TODO: Iterate through the backward list and call the backward functions (if it has parents)
        for iter in reversed(backward_list):
            if iter._parents != None:
                iter.backward_fn()
        pass



In [21]:
a_test = Node(3)
b_test = Node(2)

c_test = add_node(a_test, b_test)
f_test = mul_node(c_test, b_test)


print("Testing Node BACKWARD...")
f_test.grad = 1.0
f_test.backward()

print(f"a_grad: {a_test}")
print(f"b_grad: {b_test}")
print(f"c_grad: {c_test}")
print(f"f_grad: {f_test}")

assert a_test.grad == 2.0, "Error: BACKWARD does not update parent gradients correctly"
assert b_test.grad == 7.0, "Error: BACKWARD does not update parent gradients correctly"
assert c_test.grad == 2.0, "Error: BACKWARD does not update parent gradients correctly"
assert f_test.grad == 1.0, "Error: BACKWARD does not update parent gradients correctly"

print("All tests passed!")

Testing Node BACKWARD...
[Node(_ | data=2, grad=0), Node(_ | data=3, grad=0), Node(_ | data=5, grad=0), Node(_ | data=10, grad=1)]
a_grad: Node(_ | data=3, grad=2)
b_grad: Node(_ | data=2, grad=7)
c_grad: Node(_ | data=5, grad=2)
f_grad: Node(_ | data=10, grad=1)
All tests passed!


In [22]:
#Provided primitives

def exp_node(a_node):
    out = Node(np.exp(a_node.data), (a_node,))
    def backward_fn():
        a_node.grad += out.grad * out.data
    out.backward_fn = backward_fn
    return out

def log_node(a_node):
    out = Node(np.log(a_node.data), (a_node,))
    def backward_fn():
        a_node.grad += out.grad / a_node.data
    out.backward_fn = backward_fn
    return out

def div_node(a_node, b_node):
    out = Node(a_node.data / b_node.data, (a_node, b_node))
    def backward_fn():
        a_node.grad += out.grad / b_node.data
        b_node.grad += -out.grad * a_node.data / (b_node.data**2)
    out.backward_fn = backward_fn
    return out



In [23]:
def softplus_node(x):
    """
    softplus(x) = ln(1 + e^x)
    """
    exp = exp_node(x)
    node = Node(np.ones_like(x.data))
    add = add_node(exp, node)
    log = log_node(add)
    return log

In [24]:

# Create a test input node (use a range of values for variety)
x = Node(np.array([-2.0, -1.0, 0.0, 1.0, 2.0]))  # A mix of negative and positive values
out = softplus_node(x)
expected_forward = np.log(1.0 + np.exp(x.data))

# Print results for forward pass
print("Forward Output (softplus(x)):", out.data)
print("Expected:", expected_forward)
print("Match:", np.allclose(out.data, expected_forward))


out.grad = np.ones_like(out.data)
out.backward()
expected_grad_x = 1.0 / (1.0 + np.exp(-x.data))  # Sigmoid function

# Print results for backward pass (gradients)
print("\nGradient wrt Input:", x.grad)
print("Expected Gradient:", expected_grad_x)
print("Match:", np.allclose(x.grad, expected_grad_x))


Forward Output (softplus(x)): [0.12692801 0.31326169 0.69314718 1.31326169 2.12692801]
Expected: [0.12692801 0.31326169 0.69314718 1.31326169 2.12692801]
Match: True
[Node(_ | data=[1. 1. 1. 1. 1.], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[-2. -1.  0.  1.  2.], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[0.13533528 0.36787944 1.         2.71828183 7.3890561 ], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[1.13533528 1.36787944 2.         3.71828183 8.3890561 ], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[0.12692801 0.31326169 0.69314718 1.31326169 2.12692801], grad=[1. 1. 1. 1. 1.])]

Gradient wrt Input: [0.11920292 0.26894142 0.5        0.73105858 0.88079708]
Expected Gradient: [0.11920292 0.26894142 0.5        0.73105858 0.88079708]
Match: True


In [25]:
def leaky_relu_node(input, alpha=0.01):
    """
    Leaky ReLU activation:
        f(x) = x,         if x > 0
                alpha * x, otherwise
    """
    # Forward pass
    mask = input.data >= 0
    output = Node(np.where(mask, input.data, alpha * input.data), {input})
    def backward_fn():
        input.grad += np.where(mask, output.grad, alpha * output.grad)
    output.backward_fn = backward_fn
    return output


In [26]:
# Create a test input node
x = Node(np.array([-2.0, -1.0, 0.0, 1.0, 2.0]))  # A mix of negative and positive values
alpha = 0.01  # Leaky ReLU slope for negatives

# Compute Leaky ReLU
out = leaky_relu_node(x, alpha)

# Set the output gradient (simulating dL/dout = 1 for all elements)
out.grad = np.ones_like(out.data)

# Perform backpropagation
out.backward()

# Expected values
expected_forward = np.array([-0.02, -0.01, 0.0, 1.0, 2.0])  # Leaky ReLU applied
expected_grad_x = np.array([alpha, alpha, 1.0, 1.0, 1.0])  # Gradients from backprop

# Print results
print("Forward Output:", out.data)
print("Expected:", expected_forward)
print("Match:", np.allclose(out.data, expected_forward))

print("\nGradient wrt Input:", x.grad)
print("Expected Gradient:", expected_grad_x)
print("Match:", np.allclose(x.grad, expected_grad_x))


[Node(_ | data=[-2. -1.  0.  1.  2.], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[-0.02 -0.01  0.    1.    2.  ], grad=[1. 1. 1. 1. 1.])]
Forward Output: [-0.02 -0.01  0.    1.    2.  ]
Expected: [-0.02 -0.01  0.    1.    2.  ]
Match: True

Gradient wrt Input: [0.01 0.01 1.   1.   1.  ]
Expected Gradient: [0.01 0.01 1.   1.   1.  ]
Match: True


In [27]:
# We'll be nice and give you a div_nodes and exp_nodes primitives :)
def substract_node(a_node, b_node):
    output = Node(a_node.data - b_node.data, {a_node, b_node})
    def backward_fn():
        a_node.grad += output.grad
        b_node.grad -= output.grad
    output.backward_fn = backward_fn
    return output

def neg_node(x):
    output = Node(-x.data, {x})
    def backward_fn():
        x.grad -= output.grad
    output.backward_fn = backward_fn
    return output

def tanh_node(x):
    """
    tanh(x) = (e^x - e^-x) / (e^x + e^-x)
    """
    neg = neg_node(x)
    pos_exp = exp_node(x)
    neg_exp = exp_node(neg)
    minus = substract_node(pos_exp, neg_exp)
    add = add_node(pos_exp, neg_exp)
    div = div_node(minus, add)
    return div


In [28]:
# Create a test input node
x = Node(np.array([-2.0, -1.0, 0.0, 1.0, 2.0]), label="input")  # A mix of negative and positive values

# Compute tanh using our function
out = tanh_node(x)

# Expected forward values (using NumPy for verification)
expected_forward = np.tanh(x.data)

# Set the output gradient (simulating dL/dout = 1 for all elements)
out.grad = np.ones_like(out.data)

# Perform backpropagation
out.backward()

# # Expected gradients: d/dx tanh(x) = 1 - tanh^2(x)
expected_grad_x = 1.0 - expected_forward**2  # (1 - tanh^2(x))
print()

# Print results
print("Forward Output:", out.data)
print("Expected:", expected_forward)
print("Match:", np.allclose(out.data, expected_forward))

print("\nGradient wrt Input:", x.grad)
print("Expected Gradient:", expected_grad_x)
print("Match:", np.allclose(x.grad, expected_grad_x))


[Node(input | data=[-2. -1.  0.  1.  2.], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[ 2.  1. -0. -1. -2.], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[7.3890561  2.71828183 1.         0.36787944 0.13533528], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[0.13533528 0.36787944 1.         2.71828183 7.3890561 ], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[7.52439138 3.08616127 2.         3.08616127 7.52439138], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[-7.25372082 -2.35040239  0.          2.35040239  7.25372082], grad=[0. 0. 0. 0. 0.]), Node(_ | data=[-0.96402758 -0.76159416  0.          0.76159416  0.96402758], grad=[1. 1. 1. 1. 1.])]

Forward Output: [-0.96402758 -0.76159416  0.          0.76159416  0.96402758]
Expected: [-0.96402758 -0.76159416  0.          0.76159416  0.96402758]
Match: True

Gradient wrt Input: [0.07065082 0.41997434 1.         0.41997434 0.07065082]
Expected Gradient: [0.07065082 0.41997434 1.         0.41997434 0.07065082]
Match: True
